# 05 — Digital Finance Adoption & Financial Barriers EDA

**Project:** Indonesia Financial Inclusion & Digital Finance Analytics  
**Owner:** Mutia — digital finance & barriers; cross-review with Rafa  
**Dataset:** World Bank Global Findex 2025, Indonesia survey year **2024** (n = 1,068)  
**Input:** Private, cleaned Notebook 03 outputs; no raw ZIP or Notebook 02 session required  
**Handoff:** Approved aggregate-only tables for Notebook 06 / Google Sheets / Looker Studio

This is a reproducible, *run-from-top* notebook. Open from GitHub in Google Colab and select **Runtime → Run all**. If Notebook 03 has already saved its outputs to **MyDrive/Financial_Inclusion_Private/processed/**, this notebook will find them after Google Drive authorization.

Important: No respondent-level rows or original microdata will be printed, uploaded to GitHub, or exported as public results. Results are descriptive survey-weighted estimates, not evidence of causality or statistical significance.


## 1. Load the validated Notebook 03 pipeline outputs

All six files below must be from the **same Notebook 03 run**. In Colab, the notebook mounts your private Google Drive if needed. If running locally, place them in the repository's data/processed directory or configure CUSTOM_PROCESSED_DIR. You do **not** need to upload the original World Bank ZIP again.


In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
CUSTOM_PROCESSED_DIR = None  # Optional: absolute directory containing Notebook 03 outputs.
PRIVATE_DRIVE_DIR = Path("/content/drive/MyDrive/Financial_Inclusion_Private/processed")
NEEDED = (
    "integrated_respondents_private.csv",
    "dashboard_kpi_aggregates.csv",
    "dashboard_barriers_aggregates.csv",
    "integration_quality_checks.csv",
    "integration_manifest.json",
)
EXPECTED_ZIP_SHA256 = "fc50f0a843e289a3b9784ac81dce1018a8195388608575e6c8315c195073a64c"
EXPECTED_CSV_SHA256 = "77a50e351c16946ad107c08494efe9e8972ba628168fa385b771695971710093"

def candidate_dirs():
    paths = []
    if CUSTOM_PROCESSED_DIR is not None:
        paths.append(Path(CUSTOM_PROCESSED_DIR).expanduser())
    if IN_COLAB:
        paths.append(PRIVATE_DRIVE_DIR)
    paths.extend([
        Path.cwd() / "data" / "processed",
        Path.cwd(),
        Path("/content/Indonesia-Financial-Inclusion-Digital-Finance-Analytics/data/processed"),
    ])
    return list(dict.fromkeys(paths))

def contains_inputs(folder):
    return folder.is_dir() and all((folder / file).is_file() for file in NEEDED)

DATA_DIR = next((p for p in candidate_dirs() if contains_inputs(p)), None)
if DATA_DIR is None and IN_COLAB:
    from google.colab import drive
    print("Connecting private Google Drive for Notebook 03 outputs...")
    drive.mount("/content/drive")
    DATA_DIR = next((p for p in candidate_dirs() if contains_inputs(p)), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Notebook 03 outputs not found. Run Notebook 03 and save its outputs to "
        "MyDrive/Financial_Inclusion_Private/processed/ or set CUSTOM_PROCESSED_DIR. "
        "Required files: " + ", ".join(NEEDED)
    )

manifest03 = json.loads((DATA_DIR / "integration_manifest.json").read_text(encoding="utf-8"))
qa03 = pd.read_csv(DATA_DIR / "integration_quality_checks.csv")
df = pd.read_csv(DATA_DIR / "integrated_respondents_private.csv", low_memory=False)
canonical_kpi = pd.read_csv(DATA_DIR / "dashboard_kpi_aggregates.csv")
canonical_barriers = pd.read_csv(DATA_DIR / "dashboard_barriers_aggregates.csv")

assert manifest03.get("source_zip_sha256") == EXPECTED_ZIP_SHA256, "Unexpected raw ZIP checksum."
assert manifest03.get("source_csv_sha256") == EXPECTED_CSV_SHA256, "Unexpected raw CSV checksum."
assert manifest03.get("qa_passed") is True, "Notebook 03 integration QA did not pass."
assert len(qa03) >= 14 and qa03["status"].eq("PASS").all(), "Notebook 03 quality checks failed."
assert manifest03.get("raw_shape") == [1068, 199], "Unexpected raw dataset shape."
assert list(df.shape) == manifest03.get("clean_shape"), "Processed file does not match manifest."
assert len(df) == 1068 and set(df["year"].dropna().unique()) == {2024}
assert "wpid_random" not in df.columns, "Unexpected source respondent identifier."

print("Notebook 03 source files: VERIFIED")
print("Private input directory:", DATA_DIR)
print("Rows:", len(df), "| Canonical KPI rows:", len(canonical_kpi))
print("Notebook 03 QA:", len(qa03), "PASS")
print("No original ZIP is needed.")


## 2. Validate coding, segment labels and canonical aggregate schema

The source of truth is Notebook 03. Digital indicators use **0 = No / 1 = Yes** among valid respondents. All weighted denominators exclude invalid values. A segment with fewer than **30** valid respondents must not appear in dashboard exports. A **30–49** respondent group may be shown with a caution flag. Missing education remains missing; it is **not** a substantive group.


In [ ]:
DIGITAL_METRICS = {
    "anydigpayment": "Digital payment use",
    "internet_use": "Internet use",
    "account_mob": "Mobile money account",
    "saved": "Saved money",
    "borrowed": "Borrowed money",
}
DIMENSION_ORDER = {
    "gender": ["Female", "Male"],
    "age_group": ["15-24", "25-34", "35-44", "45-54", "55+"],
    "education": ["Primary or less", "Secondary", "Tertiary or more"],
    "income_quintile": ["Q1 - lowest", "Q2", "Q3", "Q4", "Q5 - highest"],
    "residence": ["Rural", "Urban"],
    "internet_group": ["Non-internet user", "Internet user"],
}
BARRIER_LABEL_ID = {
    "fin11a": "Lembaga keuangan terlalu jauh",
    "fin11b": "Biaya layanan keuangan terlalu mahal",
    "fin11c": "Kurangnya dokumen yang diperlukan",
    "fin11d": "Tidak memiliki cukup uang",
    "fin11e": "Anggota keluarga sudah memiliki rekening",
    "fin11f": "Kurangnya kepercayaan terhadap lembaga keuangan",
}
KPI_REQUIRED = {
    "metric", "metric_label", "dimension", "category", "sort_order",
    "numerator_wgt", "denominator_wgt", "rate_pct", "n_valid", "n_yes"
}
BARRIER_REQUIRED = {
    "rank", "barrier_code", "barrier_label", "numerator_wgt",
    "denominator_wgt", "rate_pct", "n_valid", "n_yes"
}

assert KPI_REQUIRED <= set(canonical_kpi.columns), "Notebook 03 KPI schema has changed."
assert BARRIER_REQUIRED <= set(canonical_barriers.columns), "Notebook 03 barrier schema changed."
assert set(DIGITAL_METRICS) <= set(canonical_kpi["metric"])
assert len(canonical_kpi) == 140, "Unexpected canonical KPI row count."
assert canonical_kpi["rate_pct"].between(0, 100).all()
assert canonical_kpi["n_valid"].ge(30).all(), "Small cells in canonical KPI export."
assert not canonical_kpi.duplicated(["metric", "dimension", "category"]).any()
assert canonical_kpi[["metric", "dimension", "category"]].notna().all().all()
assert set(canonical_kpi["dimension"]) == {"national", *DIMENSION_ORDER}
assert set(canonical_barriers["barrier_code"]) == set(BARRIER_LABEL_ID)
assert len(canonical_barriers) == 6
assert not canonical_barriers["barrier_code"].duplicated().any()
assert canonical_barriers["rate_pct"].between(0, 100).all()
assert canonical_barriers["n_valid"].ge(30).all()

needed_columns = {"year", "wgt", "account_fin", *DIGITAL_METRICS, *DIMENSION_ORDER, *BARRIER_LABEL_ID}
assert needed_columns <= set(df.columns), "Missing column(s) in integrated dataset."
assert df["wgt"].notna().all() and df["wgt"].gt(0).all()
for metric in DIGITAL_METRICS:
    assert df[metric].dropna().isin([0, 1]).all(), f"Invalid response code: {metric}"
for dim, allowed in DIMENSION_ORDER.items():
    unexpected = set(df[dim].dropna().unique()) - set(allowed)
    assert not unexpected, f"Unknown categories in {dim}: {unexpected}"
assert df["education"].isna().sum() == 1, "Education missingness differs from Notebook 03."
assert df["account_fin"].isin([0, 1]).all()

eligible_barriers = df.loc[df["account_fin"].eq(0)]
assert len(eligible_barriers) == 504, "Barrier eligibility count differs."
for col in BARRIER_LABEL_ID:
    assert eligible_barriers[col].dropna().isin([1, 2, 3, 4]).all(), (
        f"Unexpected barrier response code in {col}"
    )
print("Coding, segments, denominator rules and dashboard cell-size rules: VERIFIED")
print("Barrier-eligible respondents:", len(eligible_barriers))


## 3. Recalculate and verify every relevant digital KPI

For each of the **100 canonical digital-KPI rows** (5 indicators × 20 categories), this notebook recomputes weighted numerators, denominators, rates and unweighted respondent counts from Notebook 03's private integrated data. Any disagreement stops the run. These are QA results, not microdata exports.


In [ ]:
def weighted_binary_components(frame, metric):
    valid = frame.loc[frame[metric].isin([0, 1]) & frame["wgt"].gt(0)]
    if valid.empty:
        raise ValueError(f"No valid observations for {metric}")
    numerator = float((valid[metric] * valid["wgt"]).sum())
    denominator = float(valid["wgt"].sum())
    return {
        "numerator_wgt": numerator,
        "denominator_wgt": denominator,
        "rate_pct": 100.0 * numerator / denominator,
        "n_valid": int(len(valid)),
        "n_yes": int(valid[metric].eq(1).sum()),
    }

adoption = canonical_kpi.loc[
    canonical_kpi["metric"].isin(DIGITAL_METRICS)
].copy().sort_values(["metric", "dimension", "sort_order"]).reset_index(drop=True)
adoption["small_sample_warning"] = adoption["n_valid"].between(30, 49)
assert len(adoption) == 100, "Expected exactly 100 pre-approved digital KPI rows."
assert adoption["n_valid"].ge(30).all(), "Do not export groups with n < 30."
assert not adoption["category"].astype(str).str.lower().eq("nan").any()

checked = []
for rec in adoption.itertuples(index=False):
    if rec.dimension == "national":
        group = df
    else:
        assert rec.dimension in DIMENSION_ORDER
        assert rec.category in DIMENSION_ORDER[rec.dimension]
        group = df.loc[df[rec.dimension].eq(rec.category)]
    recomputed = weighted_binary_components(group, rec.metric)
    matches = (
        int(rec.n_valid) == recomputed["n_valid"]
        and int(rec.n_yes) == recomputed["n_yes"]
        and np.isclose(rec.numerator_wgt, recomputed["numerator_wgt"], rtol=0, atol=1e-5)
        and np.isclose(rec.denominator_wgt, recomputed["denominator_wgt"], rtol=0, atol=1e-5)
        and np.isclose(rec.rate_pct, recomputed["rate_pct"], rtol=0, atol=1e-5)
    )
    checked.append({
        "metric": rec.metric,
        "dimension": rec.dimension,
        "category": rec.category,
        "status": "PASS" if matches else "FAIL",
        "canonical_rate_pct": rec.rate_pct,
        "recomputed_rate_pct": recomputed["rate_pct"],
        "n_valid": recomputed["n_valid"],
    })
digital_validation = pd.DataFrame(checked)
if not digital_validation["status"].eq("PASS").all():
    display(digital_validation.loc[digital_validation["status"].eq("FAIL")])
    raise AssertionError("Notebook 05 digital KPI mismatch with Notebook 03. STOP.")
print(f"ALL {len(digital_validation)} DIGITAL KPI CROSS-CHECKS PASSED.")


## 4. National digital adoption snapshot

Rates are **survey-weighted percentages**; categories overlap and must never be stacked to total 100%.


In [ ]:
national = adoption.loc[adoption["dimension"].eq("national")].copy()
national["metric_label"] = national["metric"].map(DIGITAL_METRICS)
national["order"] = national["metric"].map({m: i for i, m in enumerate(DIGITAL_METRICS)})
national = national.sort_values("order").drop(columns="order").reset_index(drop=True)
assert len(national) == len(DIGITAL_METRICS)
BENCHMARKS = {
    "anydigpayment": 42.69,
    "internet_use": 70.46,
    "account_mob": 22.01,
    "saved": 52.99,
    "borrowed": 47.64,
}
for metric, expected in BENCHMARKS.items():
    actual = float(national.loc[national["metric"].eq(metric), "rate_pct"].iloc[0])
    assert abs(actual - expected) <= 0.02, f"Unexpected national benchmark: {metric}"
display(national[["metric", "metric_label", "rate_pct", "n_valid"]].round(3))

fig_national, ax = plt.subplots(figsize=(10, 4.6))
ax.bar(national["metric_label"], national["rate_pct"])
ax.set(title="Indonesia 2024 — National digital-finance indicators",
       ylabel="Weighted rate (%)", ylim=(0, 100))
ax.tick_params(axis="x", rotation=18)
for patch in ax.patches:
    ax.annotate(f"{patch.get_height():.1f}%",
                (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                ha="center", va="bottom")
fig_national.tight_layout()
plt.show()


## 5. Digital-payment gaps by internet use, income and demographics

Gap = first comparison group's weighted percentage minus the second group's percentage, in **percentage points (pp)**. The relationship is descriptive and does **not** establish the internet or income as the cause of digital-payment adoption.


In [ ]:
def select_metric_segment(metric, dimension):
    rows = adoption.loc[
        adoption["metric"].eq(metric) & adoption["dimension"].eq(dimension)
    ].copy().sort_values("sort_order")
    expected = DIMENSION_ORDER[dimension]
    if rows["category"].tolist() != expected:
        raise AssertionError(f"Unexpected or missing approved categories: {metric}/{dimension}")
    return rows

def segment_record(metric, dim, category):
    part = adoption.loc[
        adoption["metric"].eq(metric)
        & adoption["dimension"].eq(dim)
        & adoption["category"].eq(category)
    ]
    if len(part) != 1:
        raise ValueError(f"Expected one approved aggregate: {metric} / {dim} / {category}")
    return part.iloc[0]

gap_specs = [
    ("internet_group", "Internet user", "Non-internet user", "Internet vs non-internet"),
    ("income_quintile", "Q5 - highest", "Q1 - lowest", "Income Q5 vs Q1"),
    ("gender", "Female", "Male", "Female vs male"),
    ("residence", "Urban", "Rural", "Urban vs rural"),
]
gap_rows = []
for dim, higher, lower, title in gap_specs:
    a = segment_record("anydigpayment", dim, higher)
    b = segment_record("anydigpayment", dim, lower)
    gap_rows.append({
        "metric": "anydigpayment",
        "dimension": dim,
        "comparison": title,
        "group_a": higher, "group_b": lower,
        "rate_a_pct": float(a["rate_pct"]),
        "rate_b_pct": float(b["rate_pct"]),
        "gap_pp": float(a["rate_pct"] - b["rate_pct"]),
        "n_a": int(a["n_valid"]), "n_b": int(b["n_valid"]),
        "small_sample_warning": bool(a["n_valid"] < 50 or b["n_valid"] < 50),
    })
gaps = pd.DataFrame(gap_rows)
assert abs(gaps.loc[gaps.dimension.eq("internet_group"), "gap_pp"].iloc[0] - 36.17) < 0.03
assert abs(gaps.loc[gaps.dimension.eq("income_quintile"), "gap_pp"].iloc[0] - 33.00) < 0.03
display(gaps.round(3))


In [ ]:
internet = select_metric_segment("anydigpayment", "internet_group")
income = select_metric_segment("anydigpayment", "income_quintile")

fig_segments, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, group, title in [
    (axes[0], internet, "Digital payments by internet use"),
    (axes[1], income, "Digital payments by income quintile"),
]:
    ax.bar(group["category"], group["rate_pct"])
    ax.set(title=title, ylim=(0, 100), ylabel="Weighted rate (%)")
    ax.tick_params(axis="x", rotation=20)
    for patch, warn in zip(ax.patches, group["small_sample_warning"]):
        label = f"{patch.get_height():.1f}%" + ("*" if warn else "")
        ax.annotate(label, (patch.get_x() + patch.get_width()/2, patch.get_height()),
                    ha="center", va="bottom")
fig_segments.tight_layout()
plt.show()
print("Reminder: * denotes groups with 30–49 valid unweighted respondents.")


In [ ]:
fig_demographics, axes = plt.subplots(2, 2, figsize=(13, 8.5))
for ax, (dim, title) in zip(axes.flat, [
    ("gender", "Gender"),
    ("age_group", "Age"),
    ("education", "Education"),
    ("residence", "Urban / rural"),
]):
    rows = select_metric_segment("anydigpayment", dim)
    ax.bar(rows["category"], rows["rate_pct"])
    ax.set(title=title, ylim=(0, 100), ylabel="Weighted rate (%)")
    ax.tick_params(axis="x", rotation=20)
    for patch, warn in zip(ax.patches, rows["small_sample_warning"]):
        label = f"{patch.get_height():.1f}%" + ("*" if warn else "")
        ax.annotate(label, (patch.get_x() + patch.get_width()/2, patch.get_height()),
                    ha="center", va="bottom")
fig_demographics.suptitle("Digital-payment adoption by demographic group (*: n = 30–49)")
fig_demographics.tight_layout()
plt.show()


## 6. Financial-account barriers: verify correct eligible population and denominators

Barrier questions fin11a–fin11f apply only to respondents **without a financial-institution account** (account_fin = 0; n = 504). Each question is a separate multi-response item. Count **1 = Yes** and **2 = No** as valid; exclude codes **3 = Don't know** and **4 = Refused** from that item's denominator. Percentages do not add to 100%.

The six rates are independently recalculated here, checked against Notebook 03, and published with their correct 2024 Indonesian labels.


In [ ]:
barrier_checks = []
for rec in canonical_barriers.itertuples(index=False):
    responses = eligible_barriers.loc[
        eligible_barriers[rec.barrier_code].isin([1, 2])
    ]
    numerator = float(responses.loc[responses[rec.barrier_code].eq(1), "wgt"].sum())
    denominator = float(responses["wgt"].sum())
    if denominator <= 0:
        raise ValueError(f"No valid barrier denominator for {rec.barrier_code}")
    n_valid = int(len(responses))
    n_yes = int(responses[rec.barrier_code].eq(1).sum())
    rate = 100 * numerator / denominator
    matches = (
        int(rec.n_valid) == n_valid
        and int(rec.n_yes) == n_yes
        and np.isclose(rec.numerator_wgt, numerator, rtol=0, atol=1e-5)
        and np.isclose(rec.denominator_wgt, denominator, rtol=0, atol=1e-5)
        and np.isclose(rec.rate_pct, rate, rtol=0, atol=1e-5)
    )
    barrier_checks.append({
        "barrier_code": rec.barrier_code,
        "status": "PASS" if matches else "FAIL",
        "canonical_rate_pct": float(rec.rate_pct),
        "recomputed_rate_pct": rate,
        "n_valid": n_valid,
    })
barrier_validation = pd.DataFrame(barrier_checks)
if not barrier_validation["status"].eq("PASS").all():
    display(barrier_validation.loc[barrier_validation["status"].eq("FAIL")])
    raise AssertionError("Notebook 05 barrier mismatch with Notebook 03. STOP.")

barriers = canonical_barriers.copy().sort_values("rank").reset_index(drop=True)
barriers["barrier_label_id"] = barriers["barrier_code"].map(BARRIER_LABEL_ID)
assert barriers["barrier_label_id"].notna().all()
assert barriers.iloc[0]["barrier_code"] == "fin11d", "Unexpected leading barrier."
assert not barriers["barrier_code"].duplicated().any()
display(barriers[["rank", "barrier_code", "barrier_label_id", "rate_pct", "n_valid"]].round(3))
print(f"ALL {len(barrier_validation)} BARRIER CROSS-CHECKS PASSED.")


In [ ]:
fig_barriers, ax = plt.subplots(figsize=(11, 5.2))
plot_barriers = barriers.sort_values("rate_pct")
ax.barh(plot_barriers["barrier_label_id"], plot_barriers["rate_pct"])
ax.set(
    title="Barriers to financial-institution accounts — Indonesia 2024",
    xlabel="Weighted % of eligible valid Yes / No responses",
    xlim=(0, 100),
)
for patch, n in zip(ax.patches, plot_barriers["n_valid"]):
    ax.annotate(f"{patch.get_width():.1f}% (n={int(n)})",
                (patch.get_width(), patch.get_y()+patch.get_height()/2),
                ha="left", va="center", xytext=(4, 0), textcoords="offset points")
fig_barriers.tight_layout()
plt.show()


## 7. Final QA and dashboard-safe data exports

Output files are **aggregate-only** and saved next to Notebook 03's processed data (normally your private Google Drive). Nothing is committed to GitHub by executing this notebook.

- **person2_digital_adoption_aggregates.csv:** 100 approved canonical digital KPI rows, including single-dimension segments.
- **person2_digital_national_kpis.csv:** 5 national digital KPIs.
- **person2_digital_gap_summary.csv:** Four descriptive digital-payment comparisons.
- **person2_barrier_aggregates.csv:** 6 validated barrier rankings with English and Indonesian labels.
- **person2_notebook03_validation.csv:** 100 digital KPI + 6 barrier validation rows (QA, not for dashboard charts).
- **person2_quality_checks.csv:** Overall automated QA summary.
- **person2_eda_manifest.json:** Provenance, export file list and QA status.
- Four chart PNGs for review.

Notebook 06 must use the **same metric/dimension/category/rate_pct schema** as Notebook 03. Do not average subgroup percentages or pretend independently aggregated dimensions support combined cross-filters. If combined gender-and-income filters are desired, Notebook 06 must compute reviewed joint aggregations from the private Notebook 03 integrated file.


In [ ]:
qa_rows = [
    ("notebook03_qa", len(qa03), "at least 14 PASS", len(qa03) >= 14 and qa03.status.eq("PASS").all()),
    ("integrated_shape", str(df.shape), "(1068, 26)", df.shape == (1068, 26)),
    ("digital_kpi_row_count", len(adoption), 100, len(adoption) == 100),
    ("digital_kpi_cross_checks", int(digital_validation.status.eq("PASS").sum()), 100,
     len(digital_validation) == 100 and digital_validation.status.eq("PASS").all()),
    ("barrier_eligible_n", len(eligible_barriers), 504, len(eligible_barriers) == 504),
    ("barrier_row_count", len(barriers), 6, len(barriers) == 6),
    ("barrier_cross_checks", int(barrier_validation.status.eq("PASS").sum()), 6,
     len(barrier_validation) == 6 and barrier_validation.status.eq("PASS").all()),
    ("no_invalid_group_labels", int(adoption.category.isna().sum()), 0, adoption.category.notna().all()),
    ("no_small_cells_below_30", int(adoption.n_valid.lt(30).sum()), 0, adoption.n_valid.ge(30).all()),
    ("no_duplicate_digital_keys", int(adoption.duplicated(["metric", "dimension", "category"]).sum()),
     0, not adoption.duplicated(["metric", "dimension", "category"]).any()),
    ("national_benchmarks", len(BENCHMARKS), 5,
     all(abs(float(national.loc[national.metric.eq(m), "rate_pct"].iloc[0]) - ref) <= 0.02
         for m, ref in BENCHMARKS.items())),
]
quality05 = pd.DataFrame([
    {"check": name, "actual": actual, "expected": expected,
     "status": "PASS" if passed else "FAIL"}
    for name, actual, expected, passed in qa_rows
])
display(quality05)
if not quality05["status"].eq("PASS").all():
    raise AssertionError("Notebook 05 failed final QA: do not export!")
print(f"ALL {len(quality05)} NOTEBOOK 05 QUALITY CHECKS PASSED.")
print("All exported KPI cells have at least 30 valid respondents.")


In [ ]:
OUT_DIR = DATA_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
combined_validation = pd.concat(
    [
        digital_validation.assign(validation_type="digital_kpi"),
        barrier_validation.assign(validation_type="financial_barrier"),
    ],
    ignore_index=True,
    sort=False,
)
exports = {
    "person2_digital_adoption_aggregates.csv": adoption,
    "person2_digital_national_kpis.csv": national,
    "person2_digital_gap_summary.csv": gaps,
    "person2_barrier_aggregates.csv": barriers,
    "person2_notebook03_validation.csv": combined_validation,
    "person2_quality_checks.csv": quality05,
}
for filename, table in exports.items():
    table.to_csv(OUT_DIR / filename, index=False, float_format="%.8f")

figures = {
    "person2_national_digital.png": fig_national,
    "person2_digital_segments.png": fig_segments,
    "person2_digital_demographics.png": fig_demographics,
    "person2_financial_barriers.png": fig_barriers,
}
for filename, figure in figures.items():
    figure.savefig(OUT_DIR / filename, dpi=150, bbox_inches="tight")

manifest05 = {
    "project": "Indonesia-Financial-Inclusion-Digital-Finance-Analytics",
    "notebook": "05_eda_digital_barriers.ipynb",
    "analyst": "Mutia",
    "survey_country": "Indonesia",
    "survey_year": 2024,
    "source_zip_sha256": manifest03["source_zip_sha256"],
    "source_csv_sha256": manifest03["source_csv_sha256"],
    "input_manifest": "integration_manifest.json",
    "notebook03_qa_passed": True,
    "digital_kpi_cross_checks_passed": int(digital_validation.status.eq("PASS").sum()),
    "barrier_cross_checks_passed": int(barrier_validation.status.eq("PASS").sum()),
    "notebook05_qa_passed": True,
    "min_n_dashboard": 30,
    "export_time_utc": datetime.now(timezone.utc).isoformat(),
    "aggregate_files": list(exports),
    "chart_files": list(figures),
    "respondent_level_data_exported": False,
}
manifest_name = "person2_eda_manifest.json"
(OUT_DIR / manifest_name).write_text(
    json.dumps(manifest05, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("NOTEBOOK 05 COMPLETED: ALL CHECKS PASSED.")
print("Saved aggregate-only outputs in:", OUT_DIR)
for filename in [*exports, *figures, manifest_name]:
    print(" -", filename)
print("Do NOT commit private integrated respondents or raw survey data to GitHub.")


## 8. Interpretation and Notebook 06 handoff

**Descriptive observations to verify after running:**
- Survey-weighted digital-payment participation is approximately **42.69%**; internet usage approximately **70.46%**.
- The reported digital-payment gap between internet users and non-users is approximately **36.17 percentage points** (not a causal effect).
- The Q5–Q1 digital-payment difference is approximately **33.00 percentage points** (not a causal effect).
- Among respondents without financial-institution accounts, **not enough money** is the most commonly reported of the six listed barriers (**63.76%** of valid answers for that item). Individual barrier denominators differ due to invalid/nonresponse codes.

**Limitations:** Single-year, cross-sectional 2024 survey; weights are sampling weights, not population counts. No confidence intervals, causal inference or province ranking. Small samples (30–49 valid respondents) must be interpreted cautiously. Barriers are multi-select and not additive.

**Handoff:** Notebook 06 may use the six aggregate CSV outputs and the JSON manifest after the final QA message appears. All private input and intermediate data must stay in private storage. If any assertion fails, fix the source or definitions before exporting or publishing a dashboard.
